In [1]:
import pandas as pd
import numpy as np
import re
import string
import emoji
import unicodedata
import nltk
import spacy
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Emoji handling
import emoji

# Spell correction and contractions
from textblob import TextBlob
import contractions

# Multiprocessing and progress bars
from multiprocessing import Pool, cpu_count
from functools import lru_cache
from tqdm import tqdm

# ML and encoding
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# Deep learning tools
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Word embeddings
from gensim.models import Word2Vec
# import fasttext

In [2]:
# Setup NLP resources
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
spacy_nlp = spacy.load("en_core_web_sm")
lemmatizer = WordNetLemmatizer()

In [3]:
# Load spaCy model for NER
import en_core_web_sm
nlp_spacy = en_core_web_sm.load()

In [4]:
# Load Data
corpus_df = pd.read_csv("./Sentiment_Data.csv", encoding="ISO-8859-1")

In [5]:
# Show data overview
print("Dataset Overview:")
print(f"Dataset shape: {corpus_df.shape}")
print(f"Columns: {corpus_df.columns.tolist()}")
print("\nSentiment Distribution:")
print(corpus_df['Sentiment'].value_counts())

Dataset Overview:
Dataset shape: (451332, 2)
Columns: ['Tweet', 'Sentiment']

Sentiment Distribution:
Sentiment
Strong_Pos    233700
Neutral        77016
Mild_Pos       64004
Strong_Neg     42556
Mild_Neg       34056
Name: count, dtype: int64


### Consolidate Sentiments

In [6]:
# Merge sentiment categories
def merge_sentiments(sentiment):
    if sentiment in ['Mild_Pos', 'Strong_Pos']:
        return 'Positive'
    elif sentiment in ['Mild_Neg', 'Strong_Neg']:
        return 'Negative'
    else:
        return 'Neutral'

corpus_df['Sentiment_Merged'] = corpus_df['Sentiment'].apply(merge_sentiments)
print("\nMerged Sentiment Distribution:")
print(corpus_df['Sentiment_Merged'].value_counts())


Merged Sentiment Distribution:
Sentiment_Merged
Positive    297704
Neutral      77016
Negative     76612
Name: count, dtype: int64


In [7]:
# Balance dataset
sample_size = 10000
print(f"Balancing dataset with {sample_size} samples per class.")

def balance_dataset(df, target_col='Sentiment_Merged', min_count=None):
    if min_count is None:
        min_count = df[target_col].value_counts().min()

    balanced_dfs = []
    for sentiment in df[target_col].unique():
        sentiment_df = df[df[target_col] == sentiment]
        if len(sentiment_df) > min_count:
            balanced_df = sentiment_df.sample(n=min_count, random_state=42)
        else:
            balanced_df = sentiment_df
        balanced_dfs.append(balanced_df)

    return pd.concat(balanced_dfs, ignore_index=True)

balanced_corpus = balance_dataset(corpus_df, min_count=sample_size)
print(f"Balanced dataset shape: {balanced_corpus.shape}")

Balancing dataset with 10000 samples per class.
Balanced dataset shape: (30000, 3)


In [8]:
corpus_df = balance_dataset(corpus_df, min_count=sample_size)
print(f"Balanced dataset shape: {balanced_corpus.shape}")
print("Balanced sentiment distribution:")
print(corpus_df['Sentiment_Merged'].value_counts())

Balanced dataset shape: (30000, 3)
Balanced sentiment distribution:
Sentiment_Merged
Positive    10000
Neutral     10000
Negative    10000
Name: count, dtype: int64


In [9]:
# Create abbreviation dictionar
slang_dict = {
    "u": "you",
    "ur": "your",
    "n": "and",
    "2": "to",
    "4": "for",
    "w/": "with",
    "w/o": "without",
    "thru": "through",
    "tho": "though",
    "gonna": "going to",
    "wanna": "want to",
    "gotta": "got to",
    "kinda": "kind of",
    "sorta": "sort of",
    "outta": "out of",
    "dunno": "don't know",
    "gimme": "give me",
    "lemme": "let me",
    "btw": "by the way",
    "omg": "oh my god",
    "lol": "laugh out loud",
    "rofl": "rolling on floor laughing",
    "smh": "shaking my head",
    "tbh": "to be honest",
    "imo": "in my opinion",
    "imho": "in my humble opinion",
    "aka": "also known as",
    "asap": "as soon as possible",
    "fyi": "for your information",
    "diy": "do it yourself",
    "faq": "frequently asked questions",
    "irl": "in real life",
    "ppl": "people",
    "msg": "message",
    "txt": "text",
    "pic": "picture",
    "vid": "video",
    "app": "application",
    "tech": "technology",
    "biz": "business",
    "edu": "education",
    "gov": "government",
    "org": "organization",
    "info": "information",
    "omg": "oh my god",
    "lol": "laugh out loud",
    "btw": "by the way",
    "idk": "i do not know",
    "smh": "shaking my head",
    "afaik": "as far as i know",
    "tbh": "to be honest",
    "imo": "in my opinion",
    "icymi": "in case you missed it",
    "fwiw": "for what it is worth",
    "ftw": "for the win",
    "lmk": "let me know",
    "rn": "right now",
    "thx": "thanks",
    "til": "today i learned",
    "brb": "be right back",
    "gg": "good game",
    "noob": "newbie",
    "ootd": "outfit of the day",
    "fyp": "for you page",
    "hmu": "hit me up",
    "iiuc": "if i understand correctly",
    "ikr": "i know, right",
    "irl": "in real life",
    "iss": "i am so sorry",
    "jsyk": "just so you know",
    "lowkey": "quietly",
    "highkey": "obviously",
    "ngl": "not gonna lie",
    "oot": "out of the",
    "pls": "please",
    "rizz": "charisma",
    "ship": "support a romantic relationship",
    "slay": "do something well",
    "s/o": "shoutout",
    "stan": "support",
    "tbf": "to be fair",
    "tea": "gossip",
    "vibe check": "evaluation of mood",
    "wtf": "what the freak",
    "wym": "what you mean",
    "yaaas": "strong agreement",
    "cc": "carbon-copy",
    "cx": "correction",
    "ct": "cut tweet",
    "dm": "direct message",
    "ht": "hat tip",
    "mt": "modified tweet",
    "prt": "please retweet",
    "rt": "retweet",
    "em": "email marketing",
    "ezine": "electronic magazine",
    "fb": "facebook",
    "li": "linkedin",
    "seo": "search engine optimization",
    "sm": "social media",
    "smm": "social media marketing",
    "smo": "social media optimization",
    "sn": "social network",
    "sroi": "social return on investment",
    "ugc": "user generated content",
    "yt": "youtube",
    "ab/abt": "about",
    "b4": "before",
    "bfn": "bye for now",
    "bgd": "background",
    "bh": "blockhead",
    "br": "best regards",
    "cd9": "code 9",
    "chk": "check",
    "cul8r": "see you later",
    "dam": "don not annoy me",
    "dd": "dear daughter",
    "df": "dear fiancé",
    "dp": "profile pic",
    "ds": "dear son",
    "dyk": "did you know, do you know",
    "ema": "email address",
    "ftf": "face to face",
    "f2f": "face to face",
    "ff": "follow friday",
    "fotd": "find of the day",
    "gts": "guess the song",
    "hagn": "have a good night",
    "hand": "have a nice day",
    "hotd": "headline of the day",
    "hth": "hope that helps",
    "ic": "i see",
    "iirc": "if i remember correctly",
    "jk": "just kidding, joke",
    "jv": "joint venture",
    "kk": "ok got it",
    "kyso": "knock your socks off",
    "lhh": "laugh hella hard",
    "lmao": "laughing my ass off",
    "lo": "little one",
    "mm": "music monday",
    "mirl": "meet in real life",
    "nbd": "no big deal",
    "nct": "nobody cares, though",
    "nfw": "no freaking way",
    "njoy": "enjoy",
    "nsfw": "not safe for work",
    "nts": "note to self",
    "oh": "overheard",
    "omfg": "oh my freaking god",
    "oomf": "one of my followers",
    "orly": "oh really",
    "plmk": "please let me know",
    "pnp": "party and play",
    "qotd": "quote of the day",
    "re": "in reply to, in regards to",
    "rlrt": "real-life re-tweet",
    "rtq": "read the question",
    "sfw": "safe for work",
    "smdh": "shaking my damn head",
    "so": "significant other",
    "srs": "serious",
    "tftf": "thanks for the follow",
    "tf": "thanks for this tweet",
    "tj": "tweetjack",
    "tl": "timeline",
    "tldr": "too long, did not read",
    "tmb": "tweet me back",
    "tt": "trending topic",
    "tyia": "thank you in advance",
    "tyt": "take your time",
    "tyvw": "thank you very much",
    "w/": "with",
    "w/e": "weekend",
    "wtv": "whatever",
    "ygtr": "you got that right",
    "ykwim": "you know what i mean",
    "ykyat": "you know you are addicted to",
    "ymmv": "your mileage may vary",
    "yolo": "you only live once",
    "yoyo": "you are on your own",
    "yw": "you are welcome",
    "zomg": "omg to the max"
}
lemmatizer = WordNetLemmatizer()

### Text Cleaning Functions

In [10]:
# Process emojis
def process_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " ")) if isinstance(text, str) else ""

# Remove non-grammatical symbols
def remove_symbols(text):
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # URLs
    text = re.sub(r'@\w+', ' ', text)                   # mentions
    text = re.sub(r'#(\w+)', r'\1', text)               # hashtags keep text
    text = re.sub(r'[^a-zA-Z0-9\s\-]', ' ', text)       # non-alphanum except hyphens
    return text

# Named Entity Recognition Processing
def extract_entities(text):
    doc = spacy_nlp(text)
    ents = [ent.text for ent in doc.ents]
    return " ".join(ents) if ents else text

# Spelling corrections
def correct_spelling(text):
    return str(TextBlob(text).correct()) if isinstance(text, str) else ""

# Slang and abbreviation handler
def expand_slangs(text):
    return ' '.join([slang_dict.get(w, w) for w in text.split()])

@lru_cache(maxsize=50000)
def expand_contractions(text):
    return contractions.fix(text)

# Non-grammatical punctuation Removal
def remove_punctuations(text):
    return text.translate(str.maketrans('', '', string.punctuation))

# Tokenization and lemmatization
def tokenize_lemmatize(text):
    tokens = word_tokenize(text)
    return " ".join([lemmatizer.lemmatize(w) for w in tokens])

### Complete Pipeline

In [11]:
def clean_pipeline(text):
    text = text.lower()
    text = expand_contractions(text)
    text = process_emojis(text)
    text = remove_symbols(text)
    text = extract_entities(text)
    text = correct_spelling(text)
    text = expand_slangs(text)
    text = remove_punctuations(text)
    text = tokenize_lemmatize(text)
    return text

In [12]:
corpus_df['Tweet'] = corpus_df['Tweet'].fillna("")

In [ ]:
tqdm.pandas()
corpus_df['Cleaned_Text'] = corpus_df['Tweet'].progress_apply(clean_pipeline)

  0%|                                                                             | 23/30000 [00:14<3:09:45,  2.63it/s]

### Prepare encodings and Train-Test Splits for Models

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(corpus_df['Sentiment_Merged'])

vocab_size = 10000
max_len = 250

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(corpus_df['Cleaned_Text'])
sequences = tokenizer.texts_to_sequences(corpus_df['Cleaned_Text'])
pad_sequences_data = pad_sequences(sequences, maxlen=max_len, truncating='post')

X_train, X_test, y_train, y_test = train_test_split(
    pad_sequences_data, y, test_size=0.2, random_state=42, stratify=y)

### Compute Class Weights

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

### Save Processed Datasets & Processing Tools

In [ ]:
import pickle
import os
save_dir = "./processed_data/"
os.makedirs(save_dir, exist_ok=True)

np.save(f"{save_dir}X_train_lstm.npy", X_train)
np.save(f"{save_dir}X_test_lstm.npy", X_test)
np.save(f"{save_dir}y_train.npy", y_train)
np.save(f"{save_dir}y_test.npy", y_test)

with open(f"{save_dir}tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
with open(f"{save_dir}label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)
with open(f"{save_dir}class_weights.pkl", "wb") as f:
    pickle.dump(class_weight_dict, f)

corpus_df.to_csv(f"{save_dir}Cleaned_Dataset.csv", index=False)

### Summary Report

In [ ]:
import time
start_time = time.time()
# Run the preprocessing pipeline here
processing_time = time.time() - start_time

summary = f"""
PREPROCESSING SUMMARY
================================

Dataset Information:
- Total tweets: {len(corpus_df):,}
- Train set: {len(X_train):,}
- Test set: {len(X_test):,}
- Vocabulary size: {vocab_size:,}
- Processing time: {processing_time:.2f} seconds
- Processing speed: {len(corpus_df)/processing_time:.0f} tweets/second

Preprocessing Steps Completed:
✓ Emoji processing (converted to text)
✓ Non-grammatical symbol removal
✓ Named Entity Recognition
✓ Spell correction
✓ Abbreviation expansion
✓ Contraction expansion
✓ Punctuation cleaning
✓ Tokenization
✓ Lemmatization and stemming

Model-Ready Data Created:
✓ LSTM data: X_train_lstm.npy, X_test_lstm.npy
✓ Transformer data: X_train_transformer.npy, X_test_transformer.npy
✓ Word2Vec embeddings: w2v_embedding_matrix.npy
✓ Categorical labels: y_train.npy, y_test.npy

Ready for Models:
1. ✓ Bidirectional LSTM with learnable embeddings
2. ✓ Causal transformer with learnable embeddings
3. ✓ Causal transformer with FastText/ELMo embeddings
4. ✓ Non-causal transformer with Word2Vec embeddings
5. ✓ Non-causal transformer with learnable embeddings

Configuration saved in: tokenizer.pkl, label_encoder.pkl, class_weights.pkl
Class weights: {class_weight_dict}
"""

with open(f"{save_dir}SUMMARY.txt", "w") as f:
    f.write(summary)
print(summary)
print("PREPROCESSING COMPLETE")
print(f"All data saved to: {save_dir}")